In [0]:
%pip install seaborn torch torchvision xgboost

In [0]:
import os
import pandas as pd
import numpy as np
import mlflow.pyfunc
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torchvision import models, transforms

In [0]:
TEST_DATASET_DIR = "/Volumes/compvis_od/data/ovarian_dataset/Test_Images/"

In [0]:
test_paths = []
test_labels = []

class_names = sorted([d for d in os.listdir(TEST_DATASET_DIR) if not d.startswith(".") and os.path.isdir(os.path.join(TEST_DATASET_DIR, d))])

for class_idx, class_name in enumerate(class_names):
    folder = os.path.join(TEST_DATASET_DIR, class_name)
    for fname in os.listdir(folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            test_paths.append(os.path.join(folder, fname))
            test_labels.append(class_name)

test_df = pd.DataFrame({"image_path": test_paths})

In [0]:
MODEL_URI = "models:/CompVis_OD.data.ovarian_subtype_detector@Champion"
champion_model = mlflow.pyfunc.load_model(MODEL_URI)

# The Champion model's predict() checks `self.class_name`, but the pipeline
# only defines `self.class_names` (from the class_names.pkl artifact) — expose
# the loaded class names under the expected attribute before predicting.
pipeline = champion_model._model_impl.python_model
if not hasattr(pipeline, "class_name"):
    pipeline.class_name = pipeline.class_names

batch_size = 32
predictions = []

for i in range(0, len(test_df), batch_size):
    chunk = test_df.iloc[i : i + batch_size]
    chunk_preds = champion_model.predict(chunk)
    predictions.append(chunk_preds)

final_preds_df = pd.concat(predictions, ignore_index=True)

results_df = pd.concat([test_df, final_preds_df], axis=1)
display(results_df.head(10))

In [0]:
if test_labels is not None:
    results_df["true_subtype"] = test_labels

    acc = accuracy_score(results_df["true_subtype"], results_df["predicted_subtype"])
    macro_f1 = f1_score(results_df["true_subtype"], results_df["predicted_subtype"], average="macro", zero_division=0)

    print(f"Test Accuracy: {acc * 100:.2f}%")
    print(f"Test Macro F1: {macro_f1:.4f}\n")

    print("--- Classification Report ---")
    print(classification_report(results_df["true_subtype"], results_df["predicted_subtype"]))

    cm = confusion_matrix(results_df["true_subtype"], results_df["predicted_subtype"], labels=class_names)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Purples", xticklabels=class_names, yticklabels=class_names)
    plt.title("Holdout Test Set Confusion Matrix")
    plt.xlabel("Predicted Subtype")
    plt.ylabel("True Subtype")
    plt.tight_layout()
    plt.show()

In [0]:
spark_results = spark.createDataFrame(results_df)

(
    spark_results.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("CompVis_OD.data.test_predictions")
)